# BD PowerCast — Daily Forecast Model

## Goal

Build a date-based forecasting model for the application.

For any requested date, the system should estimate:

- Daily Energy Consumption (MWh)
- Average Demand (MW)
- Peak Demand (MW)
- Peak Demand Hour

This model will later be connected to the Streamlit application.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

import joblib
import os

In [2]:
df = pd.read_csv(
    "../data/processed/PGCB_hourly_preprocessed_stage6_continuous.csv",
    parse_dates=["datetime"]
)

print("Shape:", df.shape)
print(
    df["datetime"].min(),
    "→",
    df["datetime"].max()
)

Shape: (89101, 61)
2015-04-19 00:00:00 → 2025-06-17 12:00:00


In [3]:
daily_source = df[
    (df["row_observed"] == 1) &
    (df["demand_mw_clean"].notna())
].copy()

daily_source["date"] = (
    daily_source["datetime"].dt.date
)

daily_source["date"] = pd.to_datetime(
    daily_source["date"]
)

print(
    "Valid hourly rows:",
    len(daily_source)
)

Valid hourly rows: 88046


daily targets

In [4]:
daily_df = (
    daily_source
    .groupby("date")
    .agg(
        daily_energy_mwh=(
            "demand_mw_clean",
            "sum"
        ),

        average_demand_mw=(
            "demand_mw_clean",
            "mean"
        ),

        peak_demand_mw=(
            "demand_mw_clean",
            "max"
        ),

        hourly_records=(
            "demand_mw_clean",
            "count"
        )
    )
    .reset_index()
)

daily_df.head()

,date,daily_energy_mwh,average_demand_mw,peak_demand_mw,hourly_records
0,2015-04-19,114848.0,4993.391304,6897.0,23
1,2015-04-20,64990.0,5908.181818,6702.0,11
2,2015-04-21,99988.0,6249.250000,7473.0,16
3,2015-04-22,105857.5,4811.704545,6811.0,22
4,2015-04-23,113605.0,4733.541667,6821.0,24


days only

In [5]:
daily_df = daily_df[
    daily_df["hourly_records"] == 24
].copy()

daily_df = daily_df.reset_index(
    drop=True
)

print(
    "Complete days:",
    len(daily_df)
)

print(
    "Date range:",
    daily_df["date"].min(),
    "→",
    daily_df["date"].max()
)

Complete days: 3465
Date range: 2015-04-23 00:00:00 → 2025-06-16 00:00:00


find daily peak hour

In [6]:
peak_idx = (
    daily_source
    .groupby("date")["demand_mw_clean"]
    .idxmax()
)

peak_hours = (
    daily_source.loc[
        peak_idx,
        ["date", "datetime"]
    ]
    .copy()
)

peak_hours["peak_hour"] = (
    peak_hours["datetime"].dt.hour
)

peak_hours = peak_hours[
    ["date", "peak_hour"]
]

daily_df = daily_df.merge(
    peak_hours,
    on="date",
    how="left"
)

daily_df.head()

,date,daily_energy_mwh,average_demand_mw,peak_demand_mw,hourly_records,peak_hour
0,2015-04-23,113605.0,4733.541667,6821.0,24,18
1,2015-04-24,107629.0,4484.541667,6790.0,24,19
2,2015-04-27,128062.0,5335.916667,6323.0,24,18
3,2015-04-28,125059.0,5210.791667,6718.0,24,18
4,2015-04-29,135942.0,5664.250000,7111.0,24,19


inspect the daily dataset

In [7]:
print(daily_df.shape)

daily_df.describe()

(3465, 6)


,date,daily_energy_mwh,average_demand_mw,peak_demand_mw,hourly_records,peak_hour
count,3465,3465.000000,3465.000000,3465.000000,3465.0,3465.000000
mean,2020-06-20 12:16:00,211639.093939,8818.295581,10489.495094,24.0,17.957864
min,2015-04-23 00:00:00,88388.000000,3682.833333,5413.000000,24.0,0.000000
25%,2017-12-09 00:00:00,165478.000000,6894.916667,8483.000000,24.0,18.000000
50%,2020-07-17 00:00:00,201179.000000,8382.458333,10137.000000,24.0,19.000000
75%,2022-12-28 00:00:00,252527.000000,10521.958333,12321.000000,24.0,20.000000
max,2025-06-16 00:00:00,384650.000000,16027.083333,17200.000000,24.0,23.000000
std,NaN,58009.518508,2417.063271,2444.170610,0.0,4.578285


create date-based features

In [8]:
daily_df["year"] = daily_df["date"].dt.year
daily_df["month"] = daily_df["date"].dt.month
daily_df["day"] = daily_df["date"].dt.day
daily_df["day_of_week"] = daily_df["date"].dt.dayofweek
daily_df["day_of_year"] = daily_df["date"].dt.dayofyear

# Bangladesh: Friday = 4, Saturday = 5
daily_df["is_friday"] = (
    daily_df["day_of_week"] == 4
).astype(int)

daily_df["is_weekend"] = (
    daily_df["day_of_week"].isin([4, 5])
).astype(int)

# Continuous trend for future extrapolation
reference_date = daily_df["date"].min()

daily_df["days_since_start"] = (
    daily_df["date"] - reference_date
).dt.days

print("Date features created.")

Date features created.


cyclical seasonal features

In [9]:
# Day of week cycle
daily_df["dow_sin"] = np.sin(
    2 * np.pi * daily_df["day_of_week"] / 7
)

daily_df["dow_cos"] = np.cos(
    2 * np.pi * daily_df["day_of_week"] / 7
)

# Month cycle
daily_df["month_sin"] = np.sin(
    2 * np.pi * (daily_df["month"] - 1) / 12
)

daily_df["month_cos"] = np.cos(
    2 * np.pi * (daily_df["month"] - 1) / 12
)

# Annual seasonality
daily_df["year_sin"] = np.sin(
    2 * np.pi * daily_df["day_of_year"] / 365.25
)

daily_df["year_cos"] = np.cos(
    2 * np.pi * daily_df["day_of_year"] / 365.25
)

# Second annual harmonic
daily_df["year_sin_2"] = np.sin(
    4 * np.pi * daily_df["day_of_year"] / 365.25
)

daily_df["year_cos_2"] = np.cos(
    4 * np.pi * daily_df["day_of_year"] / 365.25
)

print("Cyclical features created.")

Cyclical features created.


define model inputs and targets

In [10]:
daily_features = [
    "days_since_start",

    "month",
    "day_of_week",

    "is_friday",
    "is_weekend",

    "dow_sin",
    "dow_cos",

    "month_sin",
    "month_cos",

    "year_sin",
    "year_cos",

    "year_sin_2",
    "year_cos_2"
]

daily_targets = [
    "daily_energy_mwh",
    "average_demand_mw",
    "peak_demand_mw"
]

print("Features:", len(daily_features))
print("Targets:", daily_targets)

Features: 13
Targets: ['daily_energy_mwh', 'average_demand_mw', 'peak_demand_mw']


chronological split

In [12]:
train_daily = daily_df[
    daily_df["date"] < "2023-01-01"
].copy()

val_daily = daily_df[
    (daily_df["date"] >= "2023-01-01") &
    (daily_df["date"] < "2024-01-01")
].copy()

test_daily = daily_df[
    daily_df["date"] >= "2024-01-01"
].copy()

print("Train:", len(train_daily))
print("Validation:", len(val_daily))
print("Test:", len(test_daily))

print("\nTrain:")
print(
    train_daily["date"].min(),
    "→",
    train_daily["date"].max()
)

print("\nValidation:")
print(
    val_daily["date"].min(),
    "→",
    val_daily["date"].max()
)

print("\nTest:")
print(
    test_daily["date"].min(),
    "→",
    test_daily["date"].max()
)

Train: 2602
Validation: 345
Test: 518

Train:
2015-04-23 00:00:00 → 2022-12-31 00:00:00

Validation:
2023-01-01 00:00:00 → 2023-12-31 00:00:00

Test:
2024-01-01 00:00:00 → 2025-06-16 00:00:00


inspect target distributions

In [13]:
print(
    daily_df[
        [
            "daily_energy_mwh",
            "average_demand_mw",
            "peak_demand_mw",
            "peak_hour"
        ]
    ].describe()
)

       daily_energy_mwh  average_demand_mw  peak_demand_mw    peak_hour
count       3465.000000        3465.000000     3465.000000  3465.000000
mean      211639.093939        8818.295581    10489.495094    17.957864
std        58009.518508        2417.063271     2444.170610     4.578285
min        88388.000000        3682.833333     5413.000000     0.000000
25%       165478.000000        6894.916667     8483.000000    18.000000
50%       201179.000000        8382.458333    10137.000000    19.000000
75%       252527.000000       10521.958333    12321.000000    20.000000
max       384650.000000       16027.083333    17200.000000    23.000000


create a future-safe trend feature

In [14]:
daily_df["trend_years"] = (
    daily_df["days_since_start"] / 365.25
)

train_daily["trend_years"] = (
    train_daily["days_since_start"] / 365.25
)

val_daily["trend_years"] = (
    val_daily["days_since_start"] / 365.25
)

test_daily["trend_years"] = (
    test_daily["days_since_start"] / 365.25
)

In [15]:
forecast_features = [
    "trend_years",

    "is_friday",
    "is_weekend",

    "dow_sin",
    "dow_cos",

    "month_sin",
    "month_cos",

    "year_sin",
    "year_cos",

    "year_sin_2",
    "year_cos_2"
]

print(
    "Number of forecasting features:",
    len(forecast_features)
)

Number of forecasting features: 11


train a trend-aware Ridge model

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

X_train = train_daily[forecast_features]
X_val = val_daily[forecast_features]

y_train_avg = train_daily[
    "average_demand_mw"
]

y_val_avg = val_daily[
    "average_demand_mw"
]

In [17]:
alpha_values = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0
]

ridge_results = []

for alpha in alpha_values:

    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=alpha)
        )
    ])

    model.fit(
        X_train,
        y_train_avg
    )

    pred = model.predict(
        X_val
    )

    mae = mean_absolute_error(
        y_val_avg,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val_avg,
            pred
        )
    )

    mape = (
        np.mean(
            np.abs(
                (
                    y_val_avg.values
                    - pred
                )
                / y_val_avg.values
            )
        )
        * 100
    )

    r2 = r2_score(
        y_val_avg,
        pred
    )

    ridge_results.append({
        "alpha": alpha,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2
    })


ridge_results = pd.DataFrame(
    ridge_results
)

ridge_results

,alpha,MAE,RMSE,MAPE,R2
0,0.01,943.985132,1218.395996,8.556103,0.634430
1,0.10,943.941942,1218.347132,8.555567,0.634459
2,1.00,943.527662,1217.889249,8.550372,0.634734
3,10.00,940.919472,1215.490731,8.513065,0.636171
4,100.00,948.236094,1234.507847,8.461656,0.624697


select the best validation model

In [18]:
best_row = ridge_results.loc[
    ridge_results["MAPE"].idxmin()
]

best_alpha = best_row["alpha"]

print(
    "Best alpha:",
    best_alpha
)

print(
    "Best validation MAPE:",
    f"{best_row['MAPE']:.4f}%"
)

Best alpha: 100.0
Best validation MAPE: 8.4617%


In [19]:
average_demand_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ridge",
        Ridge(
            alpha=best_alpha
        )
    )
])

average_demand_model.fit(
    X_train,
    y_train_avg
)

,steps,"[('scaler', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,np.float64(100.0)
,fit_intercept,True
,copy_X,True
,max_iter,None


validation metrics

In [20]:
avg_val_pred = (
    average_demand_model.predict(
        X_val
    )
)

avg_val_mae = mean_absolute_error(
    y_val_avg,
    avg_val_pred
)

avg_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_avg,
        avg_val_pred
    )
)

avg_val_mape = (
    np.mean(
        np.abs(
            (
                y_val_avg.values
                - avg_val_pred
            )
            / y_val_avg.values
        )
    )
    * 100
)

avg_val_r2 = r2_score(
    y_val_avg,
    avg_val_pred
)

print("DAILY AVERAGE DEMAND — VALIDATION")
print("=" * 40)

print(f"MAE  : {avg_val_mae:.2f} MW")
print(f"RMSE : {avg_val_rmse:.2f} MW")
print(f"MAPE : {avg_val_mape:.4f}%")
print(f"R²   : {avg_val_r2:.4f}")

DAILY AVERAGE DEMAND — VALIDATION
MAE  : 948.24 MW
RMSE : 1234.51 MW
MAPE : 8.4617%
R²   : 0.6247


train the peak-demand model

In [21]:
y_train_peak = train_daily[
    "peak_demand_mw"
]

y_val_peak = val_daily[
    "peak_demand_mw"
]

peak_results = []

for alpha in alpha_values:

    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=alpha)
        )
    ])

    model.fit(
        X_train,
        y_train_peak
    )

    pred = model.predict(
        X_val
    )

    mape = (
        np.mean(
            np.abs(
                (
                    y_val_peak.values
                    - pred
                )
                / y_val_peak.values
            )
        )
        * 100
    )

    peak_results.append({
        "alpha": alpha,
        "MAPE": mape
    })


peak_results = pd.DataFrame(
    peak_results
)

peak_results

,alpha,MAPE
0,0.01,8.456383
1,0.10,8.455971
2,1.00,8.451947
3,10.00,8.420613
4,100.00,8.316038


In [22]:
best_peak_alpha = (
    peak_results.loc[
        peak_results["MAPE"].idxmin(),
        "alpha"
    ]
)

peak_demand_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "ridge",
        Ridge(
            alpha=best_peak_alpha
        )
    )
])

peak_demand_model.fit(
    X_train,
    y_train_peak
)

peak_val_pred = (
    peak_demand_model.predict(
        X_val
    )
)

peak_val_mape = (
    np.mean(
        np.abs(
            (
                y_val_peak.values
                - peak_val_pred
            )
            / y_val_peak.values
        )
    )
    * 100
)

peak_val_mae = mean_absolute_error(
    y_val_peak,
    peak_val_pred
)

peak_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_peak,
        peak_val_pred
    )
)

peak_val_r2 = r2_score(
    y_val_peak,
    peak_val_pred
)

print("DAILY PEAK DEMAND — VALIDATION")
print("=" * 40)

print(f"MAE  : {peak_val_mae:.2f} MW")
print(f"RMSE : {peak_val_rmse:.2f} MW")
print(f"MAPE : {peak_val_mape:.4f}%")
print(f"R²   : {peak_val_r2:.4f}")

DAILY PEAK DEMAND — VALIDATION
MAE  : 1014.88 MW
RMSE : 1209.08 MW
MAPE : 8.3160%
R²   : 0.6212


In [23]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

In [24]:
seasonal_features = [
    "is_friday",
    "is_weekend",

    "dow_sin",
    "dow_cos",

    "month_sin",
    "month_cos",

    "year_sin",
    "year_cos",

    "year_sin_2",
    "year_cos_2"
]

trend_feature = [
    "trend_years"
]

In [25]:
avg_trend_model = LinearRegression()

avg_trend_model.fit(
    train_daily[trend_feature],
    train_daily["average_demand_mw"]
)

train_avg_trend = avg_trend_model.predict(
    train_daily[trend_feature]
)

val_avg_trend = avg_trend_model.predict(
    val_daily[trend_feature]
)

In [26]:
train_avg_residual = (
    train_daily["average_demand_mw"].values
    - train_avg_trend
)

In [27]:
avg_residual_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

avg_residual_model.fit(
    train_daily[seasonal_features],
    train_avg_residual
)

,n_estimators,400
,criterion,'squared_error'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [28]:
avg_residual_pred = (
    avg_residual_model.predict(
        val_daily[seasonal_features]
    )
)

avg_hybrid_val_pred = (
    val_avg_trend
    + avg_residual_pred
)

In [29]:
avg_hybrid_mae = mean_absolute_error(
    val_daily["average_demand_mw"],
    avg_hybrid_val_pred
)

avg_hybrid_rmse = np.sqrt(
    mean_squared_error(
        val_daily["average_demand_mw"],
        avg_hybrid_val_pred
    )
)

avg_hybrid_mape = (
    np.mean(
        np.abs(
            (
                val_daily["average_demand_mw"].values
                - avg_hybrid_val_pred
            )
            / val_daily["average_demand_mw"].values
        )
    )
    * 100
)

avg_hybrid_r2 = r2_score(
    val_daily["average_demand_mw"],
    avg_hybrid_val_pred
)

print("HYBRID DAILY AVERAGE DEMAND")
print("=" * 40)

print(f"MAE  : {avg_hybrid_mae:.2f} MW")
print(f"RMSE : {avg_hybrid_rmse:.2f} MW")
print(f"MAPE : {avg_hybrid_mape:.4f}%")
print(f"R²   : {avg_hybrid_r2:.4f}")

print("\nPrevious Ridge MAPE: 8.4617%")

HYBRID DAILY AVERAGE DEMAND
MAE  : 928.90 MW
RMSE : 1222.92 MW
MAPE : 8.3212%
R²   : 0.6317

Previous Ridge MAPE: 8.4617%


In [30]:
peak_trend_model = LinearRegression()

peak_trend_model.fit(
    train_daily[trend_feature],
    train_daily["peak_demand_mw"]
)

train_peak_trend = peak_trend_model.predict(
    train_daily[trend_feature]
)

val_peak_trend = peak_trend_model.predict(
    val_daily[trend_feature]
)

train_peak_residual = (
    train_daily["peak_demand_mw"].values
    - train_peak_trend
)

In [31]:
peak_residual_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

peak_residual_model.fit(
    train_daily[seasonal_features],
    train_peak_residual
)

peak_residual_pred = (
    peak_residual_model.predict(
        val_daily[seasonal_features]
    )
)

peak_hybrid_val_pred = (
    val_peak_trend
    + peak_residual_pred
)

In [32]:
peak_hybrid_mae = mean_absolute_error(
    val_daily["peak_demand_mw"],
    peak_hybrid_val_pred
)

peak_hybrid_rmse = np.sqrt(
    mean_squared_error(
        val_daily["peak_demand_mw"],
        peak_hybrid_val_pred
    )
)

peak_hybrid_mape = (
    np.mean(
        np.abs(
            (
                val_daily["peak_demand_mw"].values
                - peak_hybrid_val_pred
            )
            / val_daily["peak_demand_mw"].values
        )
    )
    * 100
)

peak_hybrid_r2 = r2_score(
    val_daily["peak_demand_mw"],
    peak_hybrid_val_pred
)

print("HYBRID DAILY PEAK DEMAND")
print("=" * 40)

print(f"MAE  : {peak_hybrid_mae:.2f} MW")
print(f"RMSE : {peak_hybrid_rmse:.2f} MW")
print(f"MAPE : {peak_hybrid_mape:.4f}%")
print(f"R²   : {peak_hybrid_r2:.4f}")

print("\nPrevious Ridge MAPE: 8.3160%")

HYBRID DAILY PEAK DEMAND
MAE  : 994.79 MW
RMSE : 1193.71 MW
MAPE : 8.1688%
R²   : 0.6307

Previous Ridge MAPE: 8.3160%


combine train + validation

In [33]:
trainval_daily = pd.concat(
    [train_daily, val_daily],
    ignore_index=True
)

print("Train + validation:", len(trainval_daily))
print(
    trainval_daily["date"].min(),
    "→",
    trainval_daily["date"].max()
)

Train + validation: 2947
2015-04-23 00:00:00 → 2023-12-31 00:00:00


final average-demand model

In [34]:
# Long-term trend
avg_trend_final = LinearRegression()

avg_trend_final.fit(
    trainval_daily[trend_feature],
    trainval_daily["average_demand_mw"]
)

trainval_avg_trend = avg_trend_final.predict(
    trainval_daily[trend_feature]
)

trainval_avg_residual = (
    trainval_daily["average_demand_mw"].values
    - trainval_avg_trend
)

# Seasonal residual model
avg_residual_final = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

avg_residual_final.fit(
    trainval_daily[seasonal_features],
    trainval_avg_residual
)

,n_estimators,400
,criterion,'squared_error'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [35]:
test_avg_trend = avg_trend_final.predict(
    test_daily[trend_feature]
)

test_avg_residual = avg_residual_final.predict(
    test_daily[seasonal_features]
)

test_avg_pred = (
    test_avg_trend
    + test_avg_residual
)

average-demand final test metrics

In [36]:
avg_test_mae = mean_absolute_error(
    test_daily["average_demand_mw"],
    test_avg_pred
)

avg_test_rmse = np.sqrt(
    mean_squared_error(
        test_daily["average_demand_mw"],
        test_avg_pred
    )
)

avg_test_mape = (
    np.mean(
        np.abs(
            (
                test_daily["average_demand_mw"].values
                - test_avg_pred
            )
            / test_daily["average_demand_mw"].values
        )
    )
    * 100
)

avg_test_r2 = r2_score(
    test_daily["average_demand_mw"],
    test_avg_pred
)

print("DAILY AVERAGE DEMAND — FINAL TEST")
print("=" * 42)
print(f"MAE  : {avg_test_mae:.2f} MW")
print(f"RMSE : {avg_test_rmse:.2f} MW")
print(f"MAPE : {avg_test_mape:.4f}%")
print(f"R²   : {avg_test_r2:.4f}")

DAILY AVERAGE DEMAND — FINAL TEST
MAE  : 1082.33 MW
RMSE : 1394.95 MW
MAPE : 9.4564%
R²   : 0.5600


In [37]:
daily_energy_prediction = test_avg_pred * 24

final peak-demand model

In [38]:
peak_trend_final = LinearRegression()

peak_trend_final.fit(
    trainval_daily[trend_feature],
    trainval_daily["peak_demand_mw"]
)

trainval_peak_trend = peak_trend_final.predict(
    trainval_daily[trend_feature]
)

trainval_peak_residual = (
    trainval_daily["peak_demand_mw"].values
    - trainval_peak_trend
)

peak_residual_final = RandomForestRegressor(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

peak_residual_final.fit(
    trainval_daily[seasonal_features],
    trainval_peak_residual
)

,n_estimators,400
,criterion,'squared_error'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [39]:
test_peak_trend = peak_trend_final.predict(
    test_daily[trend_feature]
)

test_peak_residual = peak_residual_final.predict(
    test_daily[seasonal_features]
)

test_peak_pred = (
    test_peak_trend
    + test_peak_residual
)

In [40]:
peak_test_mae = mean_absolute_error(
    test_daily["peak_demand_mw"],
    test_peak_pred
)

peak_test_rmse = np.sqrt(
    mean_squared_error(
        test_daily["peak_demand_mw"],
        test_peak_pred
    )
)

peak_test_mape = (
    np.mean(
        np.abs(
            (
                test_daily["peak_demand_mw"].values
                - test_peak_pred
            )
            / test_daily["peak_demand_mw"].values
        )
    )
    * 100
)

peak_test_r2 = r2_score(
    test_daily["peak_demand_mw"],
    test_peak_pred
)

print("DAILY PEAK DEMAND — FINAL TEST")
print("=" * 42)
print(f"MAE  : {peak_test_mae:.2f} MW")
print(f"RMSE : {peak_test_rmse:.2f} MW")
print(f"MAPE : {peak_test_mape:.4f}%")
print(f"R²   : {peak_test_r2:.4f}")

DAILY PEAK DEMAND — FINAL TEST
MAE  : 1145.77 MW
RMSE : 1370.19 MW
MAPE : 9.0479%
R²   : 0.5656


In [43]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    avg_trend_final,
    "../models/daily_avg_trend_model.pkl"
)

joblib.dump(
    avg_residual_final,
    "../models/daily_avg_residual_model.pkl"
)

joblib.dump(
    peak_trend_final,
    "../models/daily_peak_trend_model.pkl"
)

joblib.dump(
    peak_residual_final,
    "../models/daily_peak_residual_model.pkl"
)

print("Daily forecasting models saved.")

Daily forecasting models saved.


save model configuration

In [44]:
import json

daily_model_config = {
    "reference_date": str(reference_date.date()),

    "trend_feature": trend_feature,

    "seasonal_features": seasonal_features,

    "dataset_end": str(
        daily_df["date"].max().date()
    )
}

with open(
    "../models/daily_model_config.json",
    "w"
) as f:
    json.dump(
        daily_model_config,
        f,
        indent=4
    )

print("Daily model configuration saved.")

Daily model configuration saved.


create features from any date

In [45]:
def create_date_features(
    input_date,
    reference_date
):

    date = pd.Timestamp(input_date)
    reference_date = pd.Timestamp(reference_date)

    day_of_week = date.dayofweek
    month = date.month
    day_of_year = date.dayofyear

    days_since_start = (
        date - reference_date
    ).days

    trend_years = (
        days_since_start / 365.25
    )

    features = {
        "trend_years": trend_years,

        "is_friday": int(
            day_of_week == 4
        ),

        "is_weekend": int(
            day_of_week in [4, 5]
        ),

        "dow_sin": np.sin(
            2 * np.pi
            * day_of_week / 7
        ),

        "dow_cos": np.cos(
            2 * np.pi
            * day_of_week / 7
        ),

        "month_sin": np.sin(
            2 * np.pi
            * (month - 1) / 12
        ),

        "month_cos": np.cos(
            2 * np.pi
            * (month - 1) / 12
        ),

        "year_sin": np.sin(
            2 * np.pi
            * day_of_year / 365.25
        ),

        "year_cos": np.cos(
            2 * np.pi
            * day_of_year / 365.25
        ),

        "year_sin_2": np.sin(
            4 * np.pi
            * day_of_year / 365.25
        ),

        "year_cos_2": np.cos(
            4 * np.pi
            * day_of_year / 365.25
        )
    }

    return features

final predict_date() function

In [47]:
def predict_date(input_date):

    features = create_date_features(
        input_date,
        reference_date
    )

    # -----------------------------
    # TREND INPUT
    # -----------------------------

    trend_input = pd.DataFrame(
        [[
            features[
                trend_feature[0]
            ]
        ]],
        columns=trend_feature
    )

    # -----------------------------
    # SEASONAL INPUT
    # -----------------------------

    seasonal_input = pd.DataFrame(
        [[
            features[f]
            for f in seasonal_features
        ]],
        columns=seasonal_features
    )

    # -----------------------------
    # AVERAGE DEMAND
    # -----------------------------

    avg_trend = (
        avg_trend_final.predict(
            trend_input
        )[0]
    )

    avg_residual = (
        avg_residual_final.predict(
            seasonal_input
        )[0]
    )

    average_demand = (
        avg_trend
        + avg_residual
    )

    # -----------------------------
    # PEAK DEMAND
    # -----------------------------

    peak_trend = (
        peak_trend_final.predict(
            trend_input
        )[0]
    )

    peak_residual = (
        peak_residual_final.predict(
            seasonal_input
        )[0]
    )

    peak_demand = (
        peak_trend
        + peak_residual
    )

    # -----------------------------
    # DAILY ENERGY
    # -----------------------------

    daily_energy = (
        average_demand * 24
    )

    return {
        "date": pd.Timestamp(
            input_date
        ),

        "average_demand_mw":
            float(average_demand),

        "peak_demand_mw":
            float(peak_demand),

        "daily_energy_mwh":
            float(daily_energy)
    }

In [48]:
result = predict_date(
    "2026-08-10"
)

result

{'date': Timestamp('2026-08-10 00:00:00'),
 'average_demand_mw': 13972.515203854948,
 'peak_demand_mw': 15694.346926991695,
 'daily_energy_mwh': 335340.36489251873}

In [49]:
print(
    "BD POWERCAST — DAILY FORECAST"
)

print("=" * 40)

print(
    "Date:",
    result["date"].date()
)

print(
    f"Average Demand : "
    f"{result['average_demand_mw']:,.0f} MW"
)

print(
    f"Peak Demand    : "
    f"{result['peak_demand_mw']:,.0f} MW"
)

print(
    f"Daily Energy   : "
    f"{result['daily_energy_mwh']:,.0f} MWh"
)

BD POWERCAST — DAILY FORECAST
Date: 2026-08-10
Average Demand : 13,973 MW
Peak Demand    : 15,694 MW
Daily Energy   : 335,340 MWh
